# Chapter 3 — Masking NDVI by Threshold and Value Range

This notebook loads the multispectral drone imagery and **computes NDVI**, then creates threshold and range masks, compares NDVI with its mask, plots masked NDVI, and provides an interactive map–histogram explorer.

The interactive range controls drive **both** panels: pixels outside the selected NDVI interval become transparent on the map, while histogram bars outside the interval are muted.

The data are loaded exactly as in Chapter 2:

```python
ds = compute_indices(load_data())
```

`load_data()` reads the study area straight from the workshop STAC catalogue, so there is nothing to download. The NDVI formula used by the workshop code is
**NDVI = (NIR-842 − Red-668) / (NIR-842 + Red-668)**.

Keep this notebook beside `workshop_utils.py`.

## 1. Load the multispectral data and compute NDVI

`load_data()` reads the study-area window of the orthomosaic (this takes 10–20 seconds over the network). `compute_indices` selects **Red-668** and **NIR-842** and adds NDVI to the returned `xarray.Dataset`. We then take a plain NumPy array of NDVI values to work with, plus the map extent in projected coordinates (metres).

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

from workshop_utils import load_data, compute_indices, save_indices_to_tiff

plt.style.use("seaborn-v0_8-whitegrid")
NDVI_CMAP = "YlGn"
MASK_CMAP = ListedColormap(["#e8edf2", "#12664f"])

ds = compute_indices(load_data(), names=["NDVI"])
ndvi = ds["NDVI"].values
valid = np.isfinite(ndvi)

# Pixel-edge extent (left, right, bottom, top) for plotting.
half = ds.attrs["pixel_size_m"] / 2
left, right = ds.x.values.min() - half, ds.x.values.max() + half
bottom, top = ds.y.values.min() - half, ds.y.values.max() + half
extent = [left, right, bottom, top]

print(f"Shape: {ndvi.shape} | CRS: {ds.attrs['crs']} | valid pixels: {valid.sum():,}")
print(f"NDVI range: {np.nanmin(ndvi):.3f} to {np.nanmax(ndvi):.3f}")

## 2. Create a threshold mask

The default rule keeps pixels with **NDVI ≥ 0.30**. Change `threshold` or switch `KEEP_ABOVE` to `False` to retain pixels below the threshold. Invalid/no-data pixels always remain excluded.

In [ ]:
threshold = 0.30
KEEP_ABOVE = True

if KEEP_ABOVE:
    mask = valid & (ndvi >= threshold)
    rule_label = f"NDVI ≥ {threshold:.2f}"
else:
    mask = valid & (ndvi < threshold)
    rule_label = f"NDVI < {threshold:.2f}"

masked_ndvi = np.where(mask, ndvi, np.nan)
print(f"Rule: {rule_label}")
print(f"Retained: {mask.sum():,} of {valid.sum():,} valid pixels "
      f"({100 * mask.sum() / valid.sum():.1f}%)")

## 3. Plot NDVI and the Boolean mask side by side

In [ ]:
vmin, vmax = np.nanpercentile(ndvi, [2, 98])
fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)

im0 = axes[0].imshow(ndvi, cmap=NDVI_CMAP, vmin=vmin, vmax=vmax, extent=extent)
axes[0].set_title("NDVI", fontsize=15, weight="bold")
fig.colorbar(im0, ax=axes[0], label="NDVI", shrink=0.86)

im1 = axes[1].imshow(np.where(valid, mask, np.nan), cmap=MASK_CMAP, vmin=0, vmax=1,
                     extent=extent)
axes[1].set_title(f"Threshold mask: {rule_label}", fontsize=15, weight="bold")
cbar = fig.colorbar(im1, ax=axes[1], ticks=[0.25, 0.75], shrink=0.86)
cbar.ax.set_yticklabels(["Excluded", "Retained"])

for ax in axes:
    ax.set_xlabel("Easting (m)")
    ax.set_ylabel("Northing (m)")
    ax.ticklabel_format(style="plain", useOffset=False)
plt.show()

## 4. Plot the masked NDVI

Excluded pixels are transparent. The original raster is not modified.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7), constrained_layout=True)
ax.set_facecolor("#eef1f4")   # shows through the transparent (NaN) pixels
im = ax.imshow(masked_ndvi, cmap=NDVI_CMAP, vmin=vmin, vmax=vmax, extent=extent)
ax.set_title(f"Masked NDVI — retaining {rule_label}", fontsize=15, weight="bold")
ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
fig.colorbar(im, ax=ax, label="NDVI", shrink=0.88)
plt.show()

## 5. Interactive NDVI map and histogram

Drag either handle on the range slider. The map shows only pixels inside the selected interval; histogram bars in that interval are highlighted. `continuous_update=False` keeps interaction responsive.

To keep the browser quick, the map shows every *n*-th pixel (the "display stride"). Pixel counts and percentages always use the full-resolution NDVI.

In [ ]:
# Display copy of NDVI: keep every n-th pixel so the map has at most ~700 pixels per side.
display_step = int(np.ceil(max(ndvi.shape) / 700))
display_ndvi = ndvi[::display_step, ::display_step]
valid_display = np.isfinite(display_ndvi)
map_x = np.linspace(left, right, display_ndvi.shape[1])
map_y = np.linspace(top, bottom, display_ndvi.shape[0])

# Fixed histogram bins keep the histogram comparable while the range changes.
ndvi_values = ndvi[valid]
ndvi_min, ndvi_max = float(ndvi_values.min()), float(ndvi_values.max())
counts, edges = np.histogram(ndvi_values, bins=80, range=(ndvi_min, ndvi_max))
centres = (edges[:-1] + edges[1:]) / 2


def range_selection(low, high):
    # Return the display map (None outside the range), bar colours and a summary.
    inside = (display_ndvi >= low) & (display_ndvi <= high)
    map_z = np.where(inside, display_ndvi, None).tolist()   # None = transparent
    bar_colours = np.where((centres >= low) & (centres <= high), "#238b45", "#d9dee3").tolist()
    n_selected = int(((ndvi >= low) & (ndvi <= high)).sum())
    summary = f"{n_selected:,} pixels ({100 * n_selected / valid.sum():.1f}% of valid pixels)"
    return map_z, bar_colours, summary


def ndvi_map_and_histogram(figure_class=go.Figure):
    # Two-panel figure: NDVI map (trace 0) and histogram (trace 1).
    fig = figure_class(make_subplots(
        rows=1, cols=2, horizontal_spacing=0.11,
        subplot_titles=("NDVI within selected range", "NDVI distribution"),
    ))
    fig.add_trace(go.Heatmap(
        z=np.where(valid_display, display_ndvi, None).tolist(), x=map_x, y=map_y, colorscale="YlGn", zmin=ndvi_min, zmax=ndvi_max,
        colorbar=dict(title="NDVI", x=0.46, len=0.82),
        hovertemplate="x: %{x:.2f}<br>y: %{y:.2f}<br>NDVI: %{z:.3f}<extra></extra>",
    ), row=1, col=1)
    fig.add_trace(go.Bar(
        x=centres, y=counts, width=np.diff(edges), showlegend=False,
        hovertemplate="NDVI: %{x:.3f}<br>Pixels: %{y:,}<extra></extra>",
    ), row=1, col=2)
    fig.update_xaxes(title_text="Easting (m)", row=1, col=1)
    fig.update_yaxes(title_text="Northing (m)", scaleanchor="x", row=1, col=1)
    fig.update_xaxes(title_text="NDVI", row=1, col=2)
    fig.update_yaxes(title_text="Pixel count", row=1, col=2)
    fig.update_layout(template="plotly_white", width=1150, height=580,
                      margin=dict(l=65, r=40, t=100, b=60))
    return fig


slider_fig = ndvi_map_and_histogram()
slider_fig.update_layout(title=dict(text="Interactive NDVI range mask", x=0.5))

range_slider = widgets.FloatRangeSlider(
    value=(max(ndvi_min, 0.20), min(ndvi_max, 0.80)),
    min=ndvi_min, max=ndvi_max, step=0.001,
    description="NDVI range:", continuous_update=False, readout_format=".3f",
    layout=widgets.Layout(width="850px"), style={"description_width": "initial"},
)
summary = widgets.HTML()
output = widgets.Output()


def render_range(change=None):
    low, high = range_slider.value
    map_z, bar_colours, text = range_selection(low, high)
    slider_fig.data[0].z = map_z
    slider_fig.data[1].marker.color = bar_colours
    summary.value = (f"<b>Selected:</b> {text} &nbsp;|&nbsp; "
                     f"<b>Display stride:</b> {display_step}")
    with output:
        output.clear_output(wait=True)
        display(slider_fig)


range_slider.observe(render_range, names="value")
display(widgets.VBox([range_slider, summary, output]))
render_range()

## 6. Optional: drag threshold lines on the histogram

The same map and histogram, but this time you drag the **blue** (lower) and **red** (upper) lines on the histogram; the map updates to show only pixels between them. Dashed lines mark the NDVI minimum, median and maximum.

This cell uses `go.FigureWidget`, which needs the `anywidget` package (included in the workshop environment). If it does not display in your environment, use the slider in Section 5 instead — it gives the same result.

In [ ]:
ndvi_median = float(np.median(ndvi_values))
start = {"low": float(np.percentile(ndvi_values, 25)),
         "high": float(np.percentile(ndvi_values, 75))}

fig = ndvi_map_and_histogram(go.FigureWidget)
pad = 0.03 * (ndvi_max - ndvi_min)   # so lines at the min and max are not clipped
fig.update_xaxes(range=[ndvi_min - pad, ndvi_max + pad], row=1, col=2)
fig.update_layout(title=dict(text="Drag the blue and red lines to select the NDVI range", x=0.5))

# Draggable lines first, so they are fig.layout.shapes[0] and [1].
for x, colour in [(start["low"], "#1565c0"), (start["high"], "#d32f2f")]:
    fig.add_vline(x=x, line=dict(color=colour, width=5), editable=True, row=1, col=2)

# Fixed reference lines, labelled above the histogram.
for x, label, colour, dash in [(ndvi_min, "Min", "#6c757d", "dot"),
                               (ndvi_median, "Median", "#6f42c1", "dash"),
                               (ndvi_max, "Max", "#6c757d", "dot")]:
    fig.add_vline(x=x, line=dict(color=colour, width=2, dash=dash), row=1, col=2)
    fig.add_annotation(x=x, y=1, xref="x2", yref="y2 domain", yanchor="bottom",
                       text=f"{label}<br>{x:.3f}", showarrow=False, font=dict(size=11, color=colour))

# Labels that follow the draggable lines: fig.layout.annotations[-2] and [-1].
for key, colour, side in [("low", "#1565c0", "right"), ("high", "#d32f2f", "left")]:
    fig.add_annotation(x=start[key], y=0.98, xref="x2", yref="y2 domain", xanchor=side,
                       yanchor="top", showarrow=False, bgcolor="rgba(255,255,255,0.85)",
                       font=dict(size=11, color=colour))

fig._config = {"displaylogo": False, "edits": {"shapePosition": True}}

last_x = [start["low"], start["high"]]
syncing = False


def update_from_lines():
    low, high = sorted(last_x)
    map_z, bar_colours, text = range_selection(low, high)
    with fig.batch_update():
        fig.data[0].z = map_z
        fig.data[1].marker.color = bar_colours
        fig.layout.annotations[0].text = f"NDVI: {low:.3f}–{high:.3f}<br>{text}"
        for label, name, x in [(fig.layout.annotations[-2], "Lower", low),
                               (fig.layout.annotations[-1], "Upper", high)]:
            label.x = x
            label.text = f"{name}: {x:.3f}"


def on_line_moved(index):
    def handler(*args):
        # Plotly lets you tilt or stretch a line; snap it back to a full-height
        # vertical line at whichever end moved, then update the map.
        global syncing
        if syncing:
            return
        shape = fig.layout.shapes[index]
        x0, x1 = float(shape.x0), float(shape.x1)
        new_x = x0 if abs(x0 - last_x[index]) >= abs(x1 - last_x[index]) else x1
        last_x[index] = float(np.clip(new_x, ndvi_min, ndvi_max))
        syncing = True
        try:
            shape.update(x0=last_x[index], x1=last_x[index], y0=0, y1=1)
        finally:
            syncing = False
        update_from_lines()
    return handler


for i in (0, 1):
    fig.layout.shapes[i].on_change(on_line_moved(i), "x0", "x1", "y0", "y1")

update_from_lines()
display(fig)

## 7. Export the threshold result (optional)

This writes the masked NDVI from Section 2 as a georeferenced single-band GeoTIFF using `save_indices_to_tiff`. Masked pixels are stored as `NaN` (NoData); edit the output path if desired.

In [ ]:
EXPORT_MASKED_NDVI = True
OUTPUT_PATH = Path("outputs/masked_ndvi.tif")

if EXPORT_MASKED_NDVI:
    masked_ds = ds.assign(NDVI=ds["NDVI"].where(mask))   # keeps the CRS and transform
    save_indices_to_tiff(masked_ds, indices="NDVI", out_path=OUTPUT_PATH)
    print(f"Mask rule: {rule_label} ({mask.sum():,} of {valid.sum():,} valid pixels retained)")
else:
    print("Set EXPORT_MASKED_NDVI = True to write the masked raster.")

## Interpretation notes

- A threshold mask is a classification rule, not a new spectral index.
- The best NDVI threshold depends on canopy type, illumination, soil background, phenology, and the analysis objective; inspect the histogram before fixing a scientific cutoff.
- Use the Boolean mask for counts and area summaries. Use `masked_ndvi` when you need original NDVI values only for retained pixels.
- The interactive maps are downsampled for display speed, but selection statistics and exported values use the full-resolution raster.